# Parameter Profiles: Depth-Dependent Parameters

This notebook shows how to let a model parameter vary along an **auxiliary physical axis** using a *parameter profile*. The example is time-resolved XPS of a semiconductor core level: XPS is surface-sensitive but not surface-*exclusive*, so the measured spectrum is a depth average — and two of the peak parameters depend on depth:

1. **Amplitude** (`GLP_01_A`): signal from depth *z* is weighted by `exp(-z / tau)`, where `tau` is the inelastic mean free path (IMFP).
2. **Position** (`GLP_01_x0`): band bending shifts the binding energy linearly with depth, `x0(z) = x0 + m·z`.

The two ideas that drive everything here:

1. **A profile makes one parameter a function of `aux_axis`.** `file.add_par_profile(...)` attaches a profile model (from `models_profile.yaml`) to a parameter; the component is then evaluated at every `aux_axis` point and the spectra are averaged — the model of a depth-integrated measurement. The profile's parameters become regular fit parameters.
2. **Profile parameters can themselves be time-dependent.** `add_time_dependence` also searches attached profiles, so the band bending gradient `m` can carry the dynamics: photoexcitation at t = 0 collapses the band bending to flat-band condition and it recovers over ~100 ps. The depth profile then re-shapes at every time step (*spectral diffusion*) — the peak doesn't just shift, its shape changes.

Workflow:
1. **Select the data**: load (including the depth axis), inspect, set fit limits.
2. **Fit the baseline twice**: a standard fit (effective, depth-averaged values) vs. a profile-aware fit (physical, surface values) — same data, different physics.
3. **Slice-by-slice fitting** (`SbS`): track the band bending gradient through time, model-free.
4. **Profile-aware 2D fit** (`2D`): one global fit with the gradient's time dependence.
5. **Inspect the results**: compare to the known truth (`data/models_*_truth.yaml`).

In [ ]:
from pathlib import Path

import numpy as np
import trspecfit

## 1. Load and Inspect Data

### 1.1 Load Data

Load simulated time- and energy-resolved XPS data from CSV files. The data is synthetic, generated by [`data/generate_data.ipynb`](data/generate_data.ipynb) which can be re-run to regenerate or change the data.
<br><br>
The new ingredient compared to [`01_basic_fitting`](../01_basic_fitting/example.ipynb) is **`aux_axis`** — here the depth grid (nm) that profiles are evaluated on. It is *not* an axis of the `data` array (which stays `[time, energy]`); it is the integration grid for the depth average, set once on the `File` and inherited by every model loaded into it.

In [ ]:
# Create project
project = trspecfit.Project(path=Path.cwd())

# Directory containing data files
data_folder = 'data'

# Add file to project (aux_axis: depth grid for the parameter profiles)
file = trspecfit.File(
    parent_project=project,
    path=data_folder,
    data=np.loadtxt(project.path / data_folder / 'data.csv', delimiter=','),
    energy=np.loadtxt(project.path / data_folder / 'energy.csv'),
    time=np.loadtxt(project.path / data_folder / 'time.csv'),
    aux_axis=np.loadtxt(project.path / data_folder / 'aux_axis.csv'),
)

### 1.2 Inspect Data

A single core-level peak (`GLP`) on a flat background (`Offset`). At t = 0 the peak jumps toward lower binding energy, grows, and *sharpens*, then relaxes back over ~100 ps.
<br><br>
The growth and sharpening are the giveaway that this is more than a peak shift: at flat-band condition all depth layers line up at the same energy, so the depth average becomes taller and narrower. A time-dependent peak position alone (as in `01_basic_fitting`) could never reproduce that — the *shape* of the spectrum changes.

In [ ]:
# Visualize full dataset
file.describe()

### 1.3 Define Fitting Region

Set energy and time limits for the fits. They appear as black, dotted lines in the plot.

In [ ]:
# Set fitting limits (absolute values)
file.set_fit_limits(
    energy_limits=[95.5, 101.5],  # Binding energy range (eV)
    time_limits=[-50, 298]  # Time range (ps)
)

## 2. Fit Baseline / Ground State Spectrum

### 2.1 Define Baseline Spectrum

Extract the ground-state spectrum by averaging pre-excitation time slices. The Gaussian IRF (width ~10 ps) is non-causal — it bleeds the onset slightly *before* t = 0 — so stop the baseline at −30 ps, about 3 IRF widths.

In [ ]:
# Define baseline using absolute time values
file.define_baseline(
    time_start=-50,
    time_stop=-30,
    time_type='abs'  # Use absolute time values (or 'ind' for indices)
)

### 2.2 Standard Fit — What a Depth-Blind Model Sees

First fit the baseline with a plain `GLP` model (`base` in `models_energy.yaml`), no profiles. This converges happily — but every parameter is an *effective*, depth-averaged value: the amplitude bundles the IMFP weighting, the position sits below the surface binding energy (deeper, shifted layers pull it down), and the width absorbs the band bending broadening. None of them is a physical property of the sample. Keep these numbers in mind for the comparison in section 5.

In [ ]:
# Load standard baseline model (no profiles)
file.load_model(
    model_yaml='models_energy.yaml',
    model_info='base'
)

# Inspect model structure and parameters
file.describe_model(model_info='base', detail=0)

In [ ]:
# Fit standard baseline model to data
file.fit_baseline(
    model_name='base',
    stages=2,
    try_ci=0  # Confidence intervals: see 12_uncertainty_mcmc
)

### 2.3 Profile-Aware Fit — Recovering the Physics

Now load the same line shape with two depth profiles attached (`profile_base`). `file.add_par_profile(...)` works like `add_time_dependence`: it loads a profile model from a YAML file and attaches it to one parameter.

| Profile model | Target parameter | Function | Physical meaning |
|---|---|---|---|
| `IMFP` | `GLP_01_A` | `pExpDecay(z, A, tau)` | surface sensitivity |
| `BandBending` | `GLP_01_x0` | `pLinear(z, m, b)` | built-in electric field |

A profile **adds** to its base parameter at every depth point. Here `A` is fixed to 0, so the IMFP profile provides the full amplitude; `x0` stays free as the *surface* binding energy, with the band bending shift `m·z` added on top. The profile's parameters become regular fit parameters, named with the target parameter as prefix, e.g. `GLP_01_x0_pLinear_01_m`.
<br><br>
One parameter stays deliberately fixed: the IMFP `tau`. Rescaling `(A, tau, m) → (c·A, tau/c, c·m)` leaves the depth-averaged spectrum essentially unchanged, so fitting the IMFP together with the gradient is degenerate. Standard XPS practice applies: take the IMFP from tables (e.g. TPP-2M) for the material and kinetic energy, and fix it.

In [ ]:
# Load the profile-aware baseline model (GLP A = 0, fixed)
file.load_model(
    model_yaml='models_energy.yaml',
    model_info='profile_base'
)

# Attach the IMFP amplitude profile
file.add_par_profile(
    target_model='profile_base',
    target_parameter='GLP_01_A',
    profile_yaml='models_profile.yaml',
    profile_model='IMFP'
)

# Attach the band bending position profile
file.add_par_profile(
    target_model='profile_base',
    target_parameter='GLP_01_x0',
    profile_yaml='models_profile.yaml',
    profile_model='BandBending'
)

file.describe_model(model_info='profile_base', detail=0)

In [ ]:
# Visualize the profiled GLP component: one trace per depth point (deeper
# layers are weaker from the IMFP weight and shifted by the band bending),
# then the depth-averaged spectrum that is actually compared to the data.
c_GLP = file.model_active.components[1]
c_GLP.plot(plot_every=5)  # plot every 5th depth trace
c_GLP.plot(plot_traces=False)  # depth-averaged spectrum

In [ ]:
# Fit profile-aware baseline model to data
file.fit_baseline(
    model_name='profile_base',
    stages=2,
    try_ci=0  # Confidence intervals: see 12_uncertainty_mcmc
)

## 3. Slice-by-Slice Fitting

Fit each time slice independently with exactly **one** free parameter: the band bending gradient `GLP_01_x0_pLinear_01_m`. The `SbS` energy model pins all line-shape parameters, and the `IMFPfixed` profile variant pins the IMFP amplitude — all pinned values are seeded from the `profile_base` fit results.
<br><br>
The result is the gradient-vs-time trace *without* assuming any functional form — look for the jump from −0.5 to ≈ 0 eV/nm at t = 0 (flat-band condition) and the exponential recovery, with the onset smoothed by the IRF. That trace is what motivates the dynamics model in section 4.

In [ ]:
# Load Slice-by-Slice model and attach the profiles
file.load_model(
    model_yaml='models_energy.yaml',
    model_info='SbS'
)

file.add_par_profile(
    target_model='SbS',
    target_parameter='GLP_01_A',
    profile_yaml='models_profile.yaml',
    profile_model='IMFPfixed'  # pinned variant: amplitude profile stays fixed
)
file.add_par_profile(
    target_model='SbS',
    target_parameter='GLP_01_x0',
    profile_yaml='models_profile.yaml',
    profile_model='BandBending'  # gradient m is the one free parameter
)

file.describe_model(model_info='SbS', detail=0)

In [ ]:
# Perform Slice-by-Slice fit
file.fit_slice_by_slice(
    model_name='SbS',
    stages=1,
    try_ci=0  # Minimizer needs >=2 varying parameters to determine confidence intervals
)

## 4. Profile-Aware Global 2D Fitting

The missing piece is the gradient's time dependence:

```python
file.add_time_dependence(..., target_parameter='GLP_01_x0_pLinear_01_m', ...)
```

- **The target is a profile parameter.** `add_time_dependence` searches the energy model first and then every attached profile — so attach the profile *before* adding the time dependence. The model's `dim` is promoted to 2 automatically.
- **Spectral diffusion.** At every time step the depth profile is re-evaluated with the current gradient, so the whole depth-averaged line shape responds — reproducing the grow-and-sharpen transient from section 1.2.
- **What varies globally:** the surface position `x0`, the equilibrium gradient `m`, and the three dynamics parameters of `BandBendingRecovery` (`models_time.yaml`): IRF width `SD`, collapse amplitude `A`, recovery time `tau`. Line shape and IMFP amplitude stay pinned to the `profile_base` results.

In [ ]:
# Load 2D model and attach the profiles
file.load_model(
    model_yaml='models_energy.yaml',
    model_info='2D'
)

file.add_par_profile(
    target_model='2D',
    target_parameter='GLP_01_A',
    profile_yaml='models_profile.yaml',
    profile_model='IMFPfixed'  # pinned variant: amplitude profile stays fixed
)
file.add_par_profile(
    target_model='2D',
    target_parameter='GLP_01_x0',
    profile_yaml='models_profile.yaml',
    profile_model='BandBending'
)

# Add time dependence to the band bending gradient (a profile parameter)
file.add_time_dependence(
    target_model='2D',
    target_parameter='GLP_01_x0_pLinear_01_m',
    dynamics_yaml='models_time.yaml',
    dynamics_model='BandBendingRecovery'
)

print("\n=== Model with Time-Dependent Profile Parameter ===")
file.describe_model(model_info='2D', detail=1)

In [ ]:
# Perform global 2D fit
file.fit_2d(
    model_name='2D',
    stages=2,
    try_ci=0  # Skip confidence interval estimation for a faster fit
)

## 5. Inspect the Fit Results

`file.get_fit_results(fit_type='2d')` returns a `pandas.DataFrame` with one row per parameter. The names chain target parameter → profile → dynamics: `GLP_01_x0_pLinear_01_m_expFun_01_tau` reads as "the `tau` of the `expFun` dynamics on the `pLinear` gradient of the `GLP` position".

Because the data is synthetic, the results can be compared to the known truth (`data/models_*_truth.yaml`):

- The **standard** baseline fit (section 2.2) returned effective values — amplitude ≈ 11, position ≈ 98.9 eV, width ≈ 1.8 eV — none of which is a physical property of the sample (truth: surface weight 100, surface binding energy 99.5 eV, FWHM 1.2 eV).
- The **profile-aware** fits recover the physics: surface weight `pExpDecay_01_A ≈ 100`, surface binding energy `x0 ≈ 99.5 eV`, equilibrium gradient `pLinear_01_m ≈ −0.5 eV/nm`, IRF width `SD ≈ 10 ps`, collapse amplitude `expFun_01_A ≈ +0.5 eV/nm`, and recovery time `tau ≈ 100 ps`.
- Note the collapse amplitude: it cancels the equilibrium gradient at t = 0 — the fit has *measured* that the photovoltage drives the bands all the way to flat-band condition.

In [ ]:
file.get_fit_results(fit_type='2d')

## Tips for Profile Fitting

**Profile basics**
- Set `aux_axis` once on `File(...)`; every loaded model inherits it.
- A profile **adds** to its base parameter at each aux point. Fix the base parameter to 0 when the profile should provide the entire value (amplitude); keep it free when the profile is a relative shift (position).
- Vary flags of profile parameters live in the profile YAML like everywhere else — pinned variants (e.g. `IMFPfixed`) are just additional entries in `models_profile.yaml`, and pinned parameters inherit the baseline fit values.

**Identifiability**
- Depth profiles extracted from a single spectrum are strongly correlated (here: an exact scaling degeneracy between IMFP and gradient). Fix what is known independently — the IMFP from tables — and check the reported correlations before trusting profile parameters.

**Time-dependent profile parameters**
- Attach the profile first, then call `add_time_dependence` on the profile parameter (`{target_par}_{component}_{par}` name); the model is promoted to 2D automatically.

**Available profile functions** ([`src/trspecfit/functions/profile.py`](../../../src/trspecfit/functions/profile.py))
- `pExpDecay(x, A, tau)` — IMFP weighting, Beer-Lambert absorption
- `pLinear(x, m, b)` — band bending, linear gradients
- `pGauss(x, A, x0, SD)` — fluence averaging, inhomogeneous broadening

## Next Steps

- Is the profile model justified by the data? Fit both variants and compare: [`10_model_comparison`](../10_model_comparison/example.ipynb)
- Uncertainty estimation (MCMC): [`12_uncertainty_mcmc`](../12_uncertainty_mcmc/example.ipynb)
- Save, load, and export fits: [`11_save_load_export`](../11_save_load_export/example.ipynb)